# Travel Agent with Redis Cloud, Snowflake Cortex, Free Web Search & Weather API

**v2 — memory fixes**

This version keeps the same overall shape as your customised notebook (Snowflake
Cortex for chat, RedisVL for long-term vector memory, `ddgs` web search,
OpenWeatherMap, LangGraph checkpointer) but fixes the reason facts and
preferences (name, food preferences, allergies, lifestyle, etc.) weren't being
used later in the conversation.

## What was actually wrong

The original tutorial notebook (`agent_memory_tutorial.ipynb`) works because
`gpt-4o` has **native function calling** — `bind_tools()` lets the LLM reliably
decide, on its own, when to call `store_memory_tool` / `retrieve_memories_tool`.
The agent is fully "tool-based": the LLM is trusted to store and recall memories
by itself.

Snowflake Cortex (as wired up here) has **no native tool-calling API**, so the
customised notebook simulated it by asking the model to emit a JSON blob
(`{"tool": ..., "arguments": ...}`) and parsing it with regex. Two problems
followed from that:

1. **Storage was ad-hoc.** Only `name` and `pin code` had hand-written regexes
   that stored a memory. Anything else the user said — *"I'm vegetarian"*,
   *"I love hiking"*, *"my wife is allergic to shellfish"* — only had a chance
   of being stored if the LLM correctly produced the tool-call JSON on its own,
   which the transcripts show it frequently didn't (`"was []"` embed errors,
   `"I'm sorry, I couldn't process that"`, JSON getting swallowed, etc.).
2. **Retrieval was ad-hoc too.** Only "what's my name" / "what's the pin code
   of X" had a hand-written recall path. There was no general mechanism that
   fed relevant long-term memories back into the conversation, so even facts
   that *did* get stored were invisible to the model afterwards.

## The fix (matches a pattern the original notebook itself calls out)

The original notebook's markdown explicitly names the two ways to manage
memory:

> **Tool-based**: LLM decides when to store/retrieve — fewer Redis calls, may
> miss context.
> **Manual**: more Redis calls, but *"extracts more memories, providing richer
> context"*.

Because Cortex's tool-calling is unreliable, this version switches long-term
memory from tool-based to **manual/automatic**, the same tradeoff the original
notebook describes:

- **`extract_and_store_memories(...)`** — a small, structured LLM call that
  runs on *every* user turn and pulls out any fact/preference worth
  remembering (name, diet, allergies, lifestyle, past trips, budget, etc.),
  then stores it with the existing dedup logic. No more hand-written regex
  per fact type.
- **`get_memory_context(...)`** — runs before the assistant replies, does a
  vector search over the user's long-term memories, and injects the results
  into the system prompt as *"Known facts and preferences about the user"*.
  This means the assistant uses stored facts even when nothing in the message
  matches a hand-written pattern.
- Deterministic tools (date / weather / web search) keep the fast, regex-based
  routing from the customised notebook — those don't need an LLM decision and
  work fine as-is.
- `store_memory` / `retrieve_memories` are no longer offered to the chat LLM
  as JSON tool-calls (that's the unreliable path) — they're called directly
  by Python, either automatically or via an explicit `"remember that ..."`
  command.


# Cell 1 – Install dependencies

In [ ]:
# ddgs is the renamed successor to duckduckgo-search
%pip install langgraph langgraph-checkpoint redis redisvl ulid pydantic requests python-dotenv ddgs beautifulsoup4


# Cell 2 – Environment variables

In [ ]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()

def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}: ")

_set_env("SNOWFLAKE_ACCOUNT")
_set_env("SNOWFLAKE_PAT")
_set_env("MODEL")
_set_env("REDIS_URL")
_set_env("OPENWEATHER_API_KEY")   # get free key from openweathermap.org


# Cell 3 – Connect to Redis

In [ ]:
from redis import Redis
REDIS_URL = os.environ["REDIS_URL"]
redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()
print("✅ Connected to Redis")


# Cell 4 – Pydantic models

Added `Memories`, a container model used by the structured memory-extraction call (mirrors the original tutorial's `Memories` model, which it defines but never actually wires up).

In [ ]:
import ulid
from datetime import datetime
from enum import Enum
from typing import List, Optional, Union
from pydantic import BaseModel, Field


class MemoryType(str, Enum):
    """
    EPISODIC: personal experiences and user-specific preferences
              (e.g. "User is vegetarian", "User prefers window seats")
    SEMANTIC: general domain knowledge/facts
              (e.g. "Singapore requires a passport")
    """
    EPISODIC = "episodic"
    SEMANTIC = "semantic"


class Memory(BaseModel):
    """A single long-term memory."""
    content: str
    memory_type: MemoryType
    metadata: str = "{}"


class Memories(BaseModel):
    """A list of memories extracted from a message by the LLM. Wrapping the
    list in an object keeps the JSON the model has to produce simple and
    matches the shape used for structured extraction."""
    memories: List[Memory]


class StoredMemory(Memory):
    """A memory as persisted in Redis."""
    id: str
    memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
    created_at: datetime = Field(default_factory=datetime.now)
    user_id: Optional[str] = None
    thread_id: Optional[str] = None
    memory_type: Optional[MemoryType] = None

print("✅ Models ready")


# Cell 5 – RedisVL vector index

In [ ]:
from redisvl.index import SearchIndex
from redisvl.schema.schema import IndexSchema

VECTOR_DIM = 1024  # snowflake-arctic-embed-l-v2.0

memory_schema = IndexSchema.from_dict({
    "index": {"name": "agent_memories", "prefix": "memory", "key_separator": ":", "storage_type": "json"},
    "fields": [
        {"name": "content", "type": "text"},
        {"name": "memory_type", "type": "tag"},
        {"name": "metadata", "type": "text"},
        {"name": "created_at", "type": "text"},
        {"name": "user_id", "type": "tag"},
        {"name": "memory_id", "type": "tag"},
        {"name": "thread_id", "type": "tag"},
        {"name": "embedding", "type": "vector", "attrs": {"algorithm": "flat", "dims": VECTOR_DIM, "distance_metric": "cosine", "datatype": "float32"}},
    ],
})

long_term_memory_index = SearchIndex(schema=memory_schema, redis_client=redis_client, validate_on_load=True)
long_term_memory_index.create(overwrite=True)
print("✅ Long-term memory index ready")


# Cell 6 – Snowflake embedding helper

In [ ]:
import requests, json
from typing import List

ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
PAT = os.environ["SNOWFLAKE_PAT"]
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"

def embed_text(text: str) -> List[float]:
    # Safety: if text is a list, join it; guard against empty/whitespace input.
    if isinstance(text, list):
        text = " ".join(str(item) for item in text if item) if text else ""
    if not text or not isinstance(text, str) or not text.strip():
        raise ValueError("embed_text requires a non-empty string")

    url = f"https://{ACCOUNT}.snowflakecomputing.com/api/v2/cortex/inference:embed"
    headers = {
        "Authorization": f"Bearer {PAT}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    # Per Snowflake's docs (cortex-rest-api/embed-api), the required JSON field
    # is `text` — an ARRAY of strings — NOT `input`. Sending `input` silently
    # fails schema validation and Snowflake reports the (missing) `text` array
    # as size 0, which is the source of the "(was [])" error.
    payload = {"text": [text], "model": EMBED_MODEL}
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        raise RuntimeError(f"Embed API error {response.status_code}: {response.text}")

    data = response.json()["data"]
    # Response shape (see docs): data[i]["embedding"] is itself a nested list,
    # e.g. "embedding": [[-0.021, 0.005, ...]] — one inner vector per input
    # text chunk. We send exactly one text, so we want the first inner vector.
    embedding_field = data[0]["embedding"]
    return embedding_field[0] if embedding_field and isinstance(embedding_field[0], list) else embedding_field


# Cell 7 – Memory operations (store, retrieve, deduplicate)

Same logic as before, generalised (like the original tutorial) to accept a single `MemoryType` **or** a list of types when retrieving, and with stronger empty-query guards.

In [ ]:
import logging
from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")
SYSTEM_USER_ID = "system"


def similar_memory_exists(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
) -> bool:
    if not content or not content.strip():
        return False
    embedding = embed_text(content)
    filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type.value)
    if thread_id:
        filters = filters & (Tag("thread_id") == thread_id)
    q = VectorRangeQuery(
        vector=embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )
    return len(long_term_memory_index.query(q)) > 0


def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
) -> None:
    if not content or not content.strip():
        return
    if metadata is None:
        metadata = "{}"
    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info(f"Similar memory exists — skipping: {content!r}")
        return
    embedding = embed_text(content)
    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id or "",
    }
    long_term_memory_index.load([memory_data])
    logger.info(f"💾 Stored [{memory_type.value}] memory: {content!r}")


def retrieve_memories(
    query: str,
    memory_type: Union[Optional[MemoryType], List[MemoryType]] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.5,
    limit: int = 5,
) -> List[StoredMemory]:
    if not query or not query.strip():
        return []  # nothing to retrieve
    embedding = embed_text(query)
    filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]
    if memory_type:
        if isinstance(memory_type, list):
            types = "|".join(m.value for m in memory_type)
            filters.append(f"@memory_type:{{{types}}}")
        else:
            filters.append(f"@memory_type:{{{memory_type.value}}}")
    if thread_id:
        filters.append(f"@thread_id:{{{thread_id}}}")
    q = VectorRangeQuery(
        vector=embedding,
        return_fields=["content", "memory_type", "metadata", "created_at", "memory_id", "thread_id", "user_id"],
        num_results=limit,
        vector_field_name="embedding",
        distance_threshold=distance_threshold,
        dialect=2,
    )
    q.set_filter(" ".join(filters))
    results = long_term_memory_index.query(q)
    memories = []
    for doc in results:
        try:
            memories.append(StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id") or None,
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            ))
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
    return memories

print("✅ Memory operations ready")


# Cell 8 – Tools (memory, web search, weather, date)

`store_memory_tool` / `retrieve_memories_tool` are kept as plain Python
helpers (used directly by the graph and by the `"remember that ..."` fast
path) but are **no longer advertised to the chat LLM as JSON tool-calls** —
that path was the unreliable one. Deterministic tools (date / weather /
search) are still LLM-triggerable via the JSON convention since those rarely
fail to parse and don't need memory-specific correctness.

In [ ]:
from ddgs import DDGS
import re, requests, datetime

# ── Memory tools (called directly by Python, not by the LLM's JSON) ────────
def store_memory_tool(
    content: str,
    memory_type: str,
    metadata: Optional[dict] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,   # None → user-scoped, not thread-scoped
) -> str:
    try:
        mem_type = MemoryType(memory_type)
        store_memory(content, mem_type, user_id, thread_id, str(metadata) if metadata else None)
        return f"✅ Stored [{mem_type.value}] memory: {content}"
    except Exception as e:
        return f"❌ Error storing memory: {e}"

def retrieve_memories_tool(
    query: str,
    memory_type: Optional[str] = None,
    limit: int = 5,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
) -> str:
    try:
        mem_type = MemoryType(memory_type) if memory_type else None
        memories = retrieve_memories(query, mem_type, user_id, thread_id, limit=limit)
        if not memories:
            return "No relevant memories found."
        lines = ["🧠 Long-term memories:"]
        for m in memories:
            lines.append(f"  - [{m.memory_type.value}] {m.content}")
        return "\n".join(lines)
    except Exception as e:
        return f"❌ Error retrieving memories: {e}"

# ── Date tool (no external call) ──────────────────────────────────────────
def get_current_date_tool(**kwargs) -> str:
    """Return today's date and tomorrow's date."""
    now = datetime.datetime.now()
    return (
        f"Today is {now.strftime('%A, %B %d, %Y')}. "
        f"Tomorrow will be {(now + datetime.timedelta(days=1)).strftime('%A, %B %d, %Y')}."
    )

# ── Weather tool (OpenWeatherMap) ─────────────────────────────────────────
def get_weather_tool(city: str, country_code: str = "", units: str = "metric", **kwargs) -> str:
    api_key = os.environ.get("OPENWEATHER_API_KEY")
    if not api_key:
        return "ERROR: OPENWEATHER_API_KEY not set."
    q = f"{city},{country_code}" if country_code else city
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": q, "appid": api_key, "units": units}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        d = r.json()
        temp  = d["main"]["temp"]
        feels = d["main"]["feels_like"]
        cond  = d["weather"][0]["description"].capitalize()
        hum   = d["main"]["humidity"]
        wind  = d["wind"]["speed"]
        unit_sym = "C" if units == "metric" else "F"
        return (
            f"Weather in {d['name']}, {d['sys']['country']}:\n"
            f"🌡️  Temp: {temp}°{unit_sym}  (feels like {feels}°{unit_sym})\n"
            f"☁️  {cond}\n"
            f"💧 Humidity: {hum}%  💨 Wind: {wind} m/s"
        )
    except Exception as e:
        return f"Error fetching weather: {e}"

# ── Web search tool (ddgs) ─────────────────────────────────────────────────
def web_search_tool(query: str, max_results: int = 5, **kwargs) -> str:
    """
    Search the web using DuckDuckGo.
    IMPORTANT: pass the FULL query – do not strip leading keywords.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return f"No web results found for '{query}'."
        output = [f"🌐 Web search results for '{query}':"]
        for r in results:
            title   = r.get("title", "No title")
            snippet = re.sub(r"\s+", " ", r.get("body", "")).strip()
            url     = r.get("href", "")
            output.append(f"- **{title}**\n  {snippet}\n  🔗 {url}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error performing web search: {e}"

# ── Tool registry (only deterministic tools are LLM-triggerable via JSON) ──
TOOLS = [
    {
        "name": "get_current_date",
        "func": get_current_date_tool,
        "description": "Get today's and tomorrow's date. No arguments needed.",
    },
    {
        "name": "get_weather",
        "func": get_weather_tool,
        "description": "Get current weather. Parameters: city (str), country_code (optional, e.g. 'IN'), units ('metric' or 'imperial').",
    },
    {
        "name": "web_search",
        "func": web_search_tool,
        "description": "Search the web. Parameters: query (str – the FULL search query), max_results (int, default=5).",
    },
]

print("✅ Tools ready")


# Cell 9 – Snowflake Cortex LLM (JSON tool-call extraction kept for date/weather/search only)

In [ ]:
import re, json, ulid, requests, logging
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.callbacks import CallbackManagerForLLMRun
from typing import Any, List, Optional, Sequence

logger = logging.getLogger(__name__)

def extract_tool_call(text: str) -> Optional[dict]:
    """Extract a JSON object with 'tool' and 'arguments' keys, even from malformed text."""
    cleaned = re.sub(r"```(?:json)?\n?|```", "", text).strip()
    start = cleaned.find("{")
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = cleaned[start : i + 1]
                try:
                    obj = json.loads(candidate)
                    if "tool" in obj and "arguments" in obj:
                        return obj
                except json.JSONDecodeError:
                    pass
                break
    return None


def extract_json_object(text: str) -> Optional[dict]:
    """Looser variant of extract_tool_call used for memory extraction — accepts
    any well-formed top-level JSON object, not just {'tool', 'arguments'}."""
    cleaned = re.sub(r"```(?:json)?\n?|```", "", text).strip()
    start = cleaned.find("{")
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = cleaned[start : i + 1]
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    return None
    return None


class SnowflakeCortexLLM(BaseChatModel):
    account: str
    pat: str
    model: str
    temperature: float = 0.0

    def _generate(self, messages: List[BaseMessage], stop=None, run_manager=None, **kwargs) -> ChatResult:
        api_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                api_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                api_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                api_messages.append({"role": "assistant", "content": msg.content})
            elif isinstance(msg, ToolMessage):
                api_messages.append({"role": "user", "content": f"[Tool result for '{msg.name}']:\n{msg.content}"})

        url = f"https://{self.account}.snowflakecomputing.com/api/v2/cortex/v1/chat/completions"
        headers = {"Authorization": f"Bearer {self.pat}", "Content-Type": "application/json"}
        payload = {"model": self.model, "messages": api_messages, "temperature": self.temperature}

        response = requests.post(url, headers=headers, json=payload)
        if response.status_code != 200:
            raise RuntimeError(f"Snowflake API error {response.status_code}: {response.text}")

        data = response.json()
        raw_content = data["choices"][0]["message"]["content"].strip()
        logger.debug(f"Raw LLM output: {raw_content!r}")

        if raw_content in ("}", "", "{}"):
            raw_content = "I'm sorry, I couldn't process that. Could you rephrase?"

        tool_call = extract_tool_call(raw_content)
        clean_content = raw_content
        if tool_call:
            clean_content = re.sub(r"\{[\"']tool[\"'][\s\S]*?\}", "", raw_content).strip()
            if not clean_content:
                clean_content = "Using a tool to help you."

        ai_msg = AIMessage(
            content=clean_content,
            tool_calls=[
                {
                    "name": tool_call["tool"],
                    "args": tool_call["arguments"],
                    "id": f"call_{ulid.ULID()}",
                }
            ] if tool_call else [],
        )
        return ChatResult(generations=[ChatGeneration(message=ai_msg)])

    @property
    def _llm_type(self) -> str:
        return "snowflake-cortex"


account = os.environ["SNOWFLAKE_ACCOUNT"]
pat     = os.environ["SNOWFLAKE_PAT"]
model   = os.environ["MODEL"]
llm     = SnowflakeCortexLLM(account=account, pat=pat, model=model, temperature=0.0)
print("✅ Snowflake Cortex LLM ready")


# Cell 10 – Automatic memory extraction & retrieval (NEW)

This is the core fix. Two functions:

- `extract_and_store_memories(user_input, user_id)` runs on **every** user
  turn and asks the LLM to pull out anything worth remembering — name, food
  preferences, allergies, lifestyle, past trips, etc. — as structured JSON,
  then stores each fact with the existing dedup logic. This replaces the
  old approach of writing one regex per fact type.
- `get_memory_context(query, user_id)` retrieves relevant long-term memories
  and formats them for injection into the system prompt, so the assistant's
  replies are grounded in what it actually knows about the user, even when
  no hand-written pattern matches the question.

In [ ]:
MEMORY_EXTRACTION_PROMPT = """You are a memory-extraction module for a travel assistant.
Read the user's latest message and identify any NEW facts, preferences, or
personal details worth remembering for future conversations.

Look for things like:
- Personal info: name, home city / pin code, family details
- Preferences: food/dietary preferences, allergies, airline/seat preferences, budget, pace of travel
- Lifestyle: hobbies, activity level, likes/dislikes
- Past experiences: places visited, trips taken
- General/semantic travel facts the user shared (visa rules, destination facts, etc.)

Do NOT invent facts. Only extract what is explicitly stated or clearly implied.
If nothing is worth remembering, return an empty list.

Respond with ONLY a JSON object in this exact shape, no other text:
{"memories": [{"content": "<concise, self-contained fact>", "memory_type": "episodic|semantic"}]}
"""


def extract_and_store_memories(user_input: str, user_id: str, thread_id: Optional[str] = None) -> List[str]:
    """Ask the LLM to pull out any memorable facts/preferences from the
    message and store them. Runs on every turn so nothing has to match a
    hand-written pattern to be remembered."""
    if not user_input or len(user_input.strip()) < 3:
        return []
    try:
        resp = llm.invoke([
            SystemMessage(content=MEMORY_EXTRACTION_PROMPT),
            HumanMessage(content=f"User's message: {user_input}"),
        ])
        parsed = extract_json_object(resp.content or "")
        if not parsed or "memories" not in parsed:
            return []
        stored = []
        for item in parsed.get("memories") or []:
            content = (item.get("content") or "").strip()
            mtype_raw = (item.get("memory_type") or "episodic").strip().lower()
            if not content:
                continue
            try:
                mtype = MemoryType(mtype_raw)
            except ValueError:
                mtype = MemoryType.EPISODIC
            # thread_id=None → memory is scoped to the user, not one conversation,
            # so it's available across every thread the user starts.
            store_memory(content=content, memory_type=mtype, user_id=user_id, thread_id=None)
            stored.append(content)
        if stored:
            logger.info(f"🧠 Auto-extracted {len(stored)} memory(ies): {stored}")
        return stored
    except Exception as e:
        logger.error(f"Memory extraction failed: {e}")
        return []


def get_memory_context(query: str, user_id: str, limit: int = 5) -> str:
    """Retrieve relevant long-term memories and format them for prompt injection."""
    if not query or not query.strip():
        return ""
    try:
        memories = retrieve_memories(
            query=query,
            memory_type=None,
            user_id=user_id,
            thread_id=None,
            limit=limit,
            distance_threshold=0.6,
        )
        if not memories:
            return ""
        lines = [f"- {m.content}" for m in memories]
        return "Known facts and preferences about the user:\n" + "\n".join(lines)
    except Exception as e:
        logger.error(f"Memory retrieval for context failed: {e}")
        return ""

print("✅ Memory extraction & retrieval helpers ready")


# Cell 11 – Conversation persistence (transcripts + selector)

In [ ]:
from datetime import datetime
import json, logging

CONV_PREFIX = "conversation"

def save_transcript(state, thread_id: str, user_id: str) -> None:
    transcript = []
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            transcript.append({"role": "user", "content": m.content})
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                transcript.append({"role": "assistant", "content": content})
        elif isinstance(m, ToolMessage):
            transcript.append({"role": "tool", "name": m.name, "content": m.content})
        elif isinstance(m, SystemMessage):
            transcript.append({"role": "summary", "content": m.content})
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    redis_client.set(key, json.dumps(transcript))
    logger.info(f"💾 Transcript saved → {key} ({len(transcript)} turns)")

def load_transcript(thread_id: str, user_id: str) -> list:
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    raw = redis_client.get(key)
    return json.loads(raw) if raw else []

def get_all_transcript_keys(user_id: str) -> list:
    pattern = f"{CONV_PREFIX}:{user_id}:*"
    return sorted([k.decode() for k in redis_client.keys(pattern)])

def select_conversation(user_id: str = "demo_user") -> tuple:
    """Interactive selector; returns (thread_id, transcript)."""
    keys = get_all_transcript_keys(user_id)
    print("\n" + "=" * 55)
    print("  📚 CONVERSATION SELECTOR")
    print("=" * 55)
    if not keys:
        print("  No previous conversations found. Starting new.\n")
        thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        return thread_id, []
    print("  [0] 🆕 Start a new conversation\n")
    for i, key in enumerate(keys, start=1):
        thread_part = key.split(":", 2)[2] if key.count(":") >= 2 else key
        raw = redis_client.get(key)
        turns = len(json.loads(raw)) if raw else 0
        preview = ""
        if raw:
            for turn in json.loads(raw):
                if turn.get("role") == "user":
                    preview = turn.get("content", "")[:60]
                    break
        print(f"  [{i}] 🗂  {thread_part}")
        print(f"       {turns} turns  |  \"{preview}{'...' if len(preview) == 60 else ''}\"")
        print()
    print("=" * 55)
    while True:
        try:
            choice = input(f"  Choose [0-{len(keys)}]: ").strip()
        except (EOFError, KeyboardInterrupt):
            choice = "0"
        if choice == "0":
            thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            print(f"\n  🆕 New conversation started: {thread_id}\n")
            return thread_id, []
        if choice.isdigit() and 1 <= int(choice) <= len(keys):
            idx = int(choice) - 1
            chosen_key = keys[idx]
            thread_id = chosen_key.split(":", 2)[2]
            transcript = json.loads(redis_client.get(chosen_key))
            print(f"\n  📂 Resuming: {thread_id} ({len(transcript)} turns)\n")
            return thread_id, transcript
        print(f"  ❌ Invalid choice. Enter a number between 0 and {len(keys)}.")

def print_transcript(thread_id: str, user_id: str = "demo_user") -> None:
    transcript = load_transcript(thread_id, user_id)
    if not transcript:
        print(f"No transcript found for user={user_id!r} thread={thread_id!r}")
        return
    print(f"\n{'='*60}")
    print(f"TRANSCRIPT  user={user_id}  thread={thread_id}")
    print(f"{'='*60}")
    for turn in transcript:
        role = turn["role"].upper()
        name = f" ({turn['name']})" if turn.get("name") else ""
        print(f"\n[{role}{name}]\n{turn.get('content', '')}")
    print()

print("✅ Conversation transcript helpers ready")


# Cell 12 – LangGraph workflow

**Changes vs the customised notebook:**

1. `respond_to_user` now calls `extract_and_store_memories(...)` unconditionally
   at the top, before any routing — so facts get captured even when the same
   message also triggers a weather/date/search lookup.
2. The old name-only / pin-code-only recall regexes are replaced by a
   **generic personal-question fast path** that pulls relevant memories with
   `get_memory_context(...)` and lets the LLM phrase the answer — this is what
   makes food preferences, allergies, lifestyle, etc. usable, not just name and
   pin code.
3. The LLM fallback (for genuine open-ended conversation) now always injects
   `get_memory_context(...)` into the system prompt, so *any* reply — not just
   ones matching a regex — is grounded in what's known about the user.
4. The chat LLM is no longer asked to emit tool-call JSON for `store_memory` /
   `retrieve_memories` — only for the three deterministic tools — since that
   JSON path was the unreliable one for memory.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.redis import RedisSaver
from langchain_core.runnables.config import RunnableConfig
import re
from datetime import datetime, timedelta

class RuntimeState(MessagesState):
    tool_call_count: int

MAX_TOOL_CALLS_PER_TURN = 3

SYSTEM_PROMPT = """You are a friendly, knowledgeable travel assistant.

You may be given a list of known facts and preferences about the user below
your instructions — use them naturally when relevant, and don't ask the user
to repeat information you already have.

Available tools:
1. get_current_date() – no args.
2. get_weather(city, country_code, units) – country_code optional, units 'metric' or 'imperial'.
3. web_search(query) – search the web; pass the FULL query string.

To call a tool, respond with ONLY this JSON (no other text):
{"tool": "tool_name", "arguments": {"arg1": "value"}}

If you don't need a tool, reply in plain text.
"""

# ─────────────────────────────────────────────────────────────────────────
# Helper: does the message look like a pure date question?
# ─────────────────────────────────────────────────────────────────────────
_DATE_ONLY_RE = re.compile(
    r"^(?:what(?:'s|\s+is)?\s+(?:the\s+)?|"
    r"tell\s+me\s+(?:the\s+)?)?(?:today|tomorrow|current\s+date|today\'s\s+date|date\s+today)\s*\?*$",
    re.IGNORECASE,
)

# ─────────────────────────────────────────────────────────────────────────
# Helper: generic "recall something about me" question?
# ─────────────────────────────────────────────────────────────────────────
_RECALL_RE = re.compile(
    r"(what(?:'s|\s+is)?\s+my\s+\w+"
    r"|do\s+you\s+(?:know|remember)\s+(?:my|about\s+me)"
    r"|what\s+do\s+you\s+know\s+about\s+me"
    r"|do\s+i\s+(?:have|like|prefer)"
    r"|am\s+i\s+allerg"
    r"|what\s+are\s+my\s+\w+)",
    re.IGNORECASE,
)

# ─────────────────────────────────────────────────────────────────────────
# Helper: build a normalised search query from user input.
# ─────────────────────────────────────────────────────────────────────────
_SEARCH_TRIGGER_RE = re.compile(
    r"^(?:(?:do\s+(?:a|the)\s+)?(?:web\s*)?search\s+for\s+(?:the\s+)?"
    r"|find(?:s+me)?\s+(?:information\s+(?:about|on)\s+)?"
    r"|look(?:s+it)?\s+up\s+"
    r"|search(?:s+for)?\s+"
    r"|news\s+(?:on|about|for)?\s*"
    r"|tell\s+me\s+about\s+"
    r"|what\s+(?:is|are|do\s+you\s+know\s+about)\s+)",
    re.IGNORECASE,
)

def _build_search_query(user_input: str) -> Optional[str]:
    stripped = user_input.strip()
    m = _SEARCH_TRIGGER_RE.match(stripped)
    if m:
        remainder = stripped[m.end():].strip(" ?.")
        if remainder:
            return remainder
    return None

# ─────────────────────────────────────────────────────────────────────────
# Agent node
# ─────────────────────────────────────────────────────────────────────────
def respond_to_user(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    user_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not user_msgs:
        return state

    user_input   = user_msgs[-1].content.strip()
    configurable = config.get("configurable", {}) if config else {}
    user_id      = configurable.get("user_id", SYSTEM_USER_ID)
    thread_id    = configurable.get("thread_id")

    # ── 0. Always try to learn something from this message ────────────────
    # Runs before any routing so facts are captured even on messages that are
    # ALSO a weather/date/search request (e.g. "weather in Goa, I'm vegan").
    extract_and_store_memories(user_input, user_id=user_id)

    # ── 1. Explicit "remember ..." command (fast path, no LLM round-trip) ──
    remember_match = re.match(r"^remember\s+(?:that\s+)?(.+)$", user_input, re.IGNORECASE)
    if remember_match:
        fact = remember_match.group(1).strip()
        store_memory_tool(content=fact, memory_type="episodic", user_id=user_id)
        state["messages"].append(AIMessage(content=f"✅ Got it, I'll remember: {fact}"))
        return state

    # ── 2. Name-set shortcut (kept for a snappy, LLM-free acknowledgement) ─
    store_name = re.search(r"(?:my name is|i am|call me)\s+(\w+)", user_input, re.IGNORECASE)
    if store_name:
        name = store_name.group(1).capitalize()
        state["messages"].append(AIMessage(content=f"✅ Nice to meet you, **{name}**! I'll remember that."))
        return state

    # ── 3. Generic recall ("what's my X", "do I have any allergies", ...) ─
    #     This replaces the old name-only / pin-code-only regexes: ANY fact
    #     that got stored (by step 0, on this turn or a previous one) can be
    #     surfaced here, not just name and pin code.
    if _RECALL_RE.search(user_input):
        context = get_memory_context(user_input, user_id=user_id, limit=5)
        if context:
            answer = llm.invoke([
                SystemMessage(content="Answer the user's question about themselves using ONLY the facts below. Be warm and concise. If the facts don't answer the question, say you don't have that saved."),
                HumanMessage(content=f"{context}\n\nUser's question: {user_input}"),
            ])
            reply = (answer.content or "").strip() or "Here's what I know about you."
        else:
            reply = "I don't have that saved yet — feel free to tell me!"
        state["messages"].append(AIMessage(content=reply))
        return state

    # ── 4. Date — only for pure date questions ────────────────────────────
    if _DATE_ONLY_RE.match(user_input.strip()):
        now      = datetime.now()
        tomorrow = now + timedelta(days=1)
        answer   = (
            f"Today is **{now.strftime('%A, %B %d, %Y')}**. "
            f"Tomorrow is **{tomorrow.strftime('%A, %B %d, %Y')}**."
        )
        state["messages"].append(AIMessage(content=answer))
        return state

    # ── 5. Weather ────────────────────────────────────────────────────────
    weather_match = re.search(
        r"weather(?:s+(?:in|of|for|at))?\s+([\w\s]+?)(?:\s*,\s*(\w{2}))?\s*\?*$",
        user_input,
        re.IGNORECASE,
    )
    if weather_match:
        city    = weather_match.group(1).strip()
        country = (weather_match.group(2) or "").strip()
        result  = get_weather_tool(city=city, country_code=country, units="metric")
        state["messages"].append(AIMessage(content=result))
        return state

    # ── 6. Explicit web / info search ─────────────────────────────────────
    search_query = _build_search_query(user_input)
    if search_query:
        if re.search(r"news", user_input, re.IGNORECASE):
            search_query = f"{search_query} news 2026"
        result = web_search_tool(query=search_query, max_results=5)
        state["messages"].append(AIMessage(content=result))
        return state

    # ── 7. LLM fallback for genuine conversation, grounded with memories ───
    tool_calls_this_turn = state.get("tool_call_count", 0)
    memory_context = get_memory_context(user_input, user_id=user_id, limit=5)
    system_content = SYSTEM_PROMPT
    if memory_context:
        system_content += f"\n\n{memory_context}"

    llm_messages = [SystemMessage(content=system_content)]
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            llm_messages.append(HumanMessage(content=m.content))
        elif isinstance(m, AIMessage):
            if m.content and not extract_tool_call(m.content):
                llm_messages.append(AIMessage(content=m.content))
        elif isinstance(m, ToolMessage):
            llm_messages.append(HumanMessage(content=f"[Tool result for '{m.name}']:\n{m.content}"))

    if tool_calls_this_turn >= MAX_TOOL_CALLS_PER_TURN:
        llm_messages.append(HumanMessage(content="You have used the maximum tool calls. Reply directly in plain text."))

    ai_msg = llm.invoke(llm_messages)

    if not ai_msg.content or ai_msg.content.strip() in ("}", ""):
        logger.warning("Malformed LLM response — falling back to web search on user query.")
        result = web_search_tool(query=user_input, max_results=5)
        state["messages"].append(AIMessage(content=result))
        return state

    state["messages"].append(ai_msg)
    return state


# ── execute_tools node ────────────────────────────────────────────────────
def execute_tools(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    ai_msgs = [m for m in state["messages"] if isinstance(m, AIMessage) and getattr(m, "tool_calls", None)]
    if not ai_msgs:
        return state

    latest       = ai_msgs[-1]
    configurable = config.get("configurable", {}) if config else {}
    user_id      = configurable.get("user_id", SYSTEM_USER_ID)

    for tc in latest.tool_calls:
        tool_name = tc["name"]
        tool_args = dict(tc["args"])
        tool_func = next((t["func"] for t in TOOLS if t["name"] == tool_name), None)

        if not tool_func:
            result = f"Tool '{tool_name}' not found."
        else:
            try:
                result = tool_func(**tool_args)
                logger.info(f"Tool '{tool_name}' result: {result!r}")
            except Exception as e:
                result = f"Error executing tool '{tool_name}': {e}"
                logger.error(result)

        state["messages"].append(ToolMessage(content=str(result), tool_call_id=tc["id"], name=tool_name))

    state["tool_call_count"] = state.get("tool_call_count", 0) + 1
    return state


# ── summarize node ────────────────────────────────────────────────────────
MESSAGE_THRESHOLD = 8

def summarize_conversation(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    messages = state["messages"]
    if len(messages) < MESSAGE_THRESHOLD:
        return state

    readable = []
    for m in messages:
        if isinstance(m, HumanMessage):
            readable.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                readable.append(f"Assistant: {content}")

    if not readable:
        return state

    summary_prompt = (
        "Summarise this travel assistant conversation in 3–5 sentences. "
        "Focus on destinations, user preferences, and decisions made.\n\n"
        + "\n".join(readable)
    )
    summary_msg = llm.invoke([
        SystemMessage(content="You are a concise conversation summariser."),
        HumanMessage(content=summary_prompt),
    ])
    logger.info(f"Summarised {len(messages)} messages.")

    summary_sys = SystemMessage(
        content=f"Summary of conversation so far:\n\n{summary_msg.content}\n\nContinue from here."
    )
    keep = messages[-2:] if len(messages) >= 2 else messages
    state["messages"] = [summary_sys] + keep
    return state


# ── routing ───────────────────────────────────────────────────────────────
def decide_next(state: RuntimeState) -> str:
    last = state["messages"][-1] if state["messages"] else None
    if state.get("tool_call_count", 0) >= MAX_TOOL_CALLS_PER_TURN:
        return "summarize"
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "execute_tools"
    return "summarize"


# ── compile graph ─────────────────────────────────────────────────────────
workflow = StateGraph(RuntimeState)
workflow.add_node("agent", respond_to_user)
workflow.add_node("execute_tools", execute_tools)
workflow.add_node("summarize", summarize_conversation)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    decide_next,
    {"execute_tools": "execute_tools", "summarize": "summarize"},
)
workflow.add_edge("execute_tools", "agent")
workflow.add_edge("summarize", END)

redis_saver = RedisSaver(redis_client=redis_client)
redis_saver.setup()
graph = workflow.compile(checkpointer=redis_saver)

print("✅ Graph compiled")


# Cell 13 – Interactive loop (with selector and commands)

`memories` now also shows which are `episodic` vs `semantic` and is otherwise unchanged. `debug` prints the injected memory context for the current turn so you can see what the assistant was grounded on.

In [ ]:
from langchain_core.messages import HumanMessage

def main(user_id: str = "demo_user", verbose: bool = False):

    # ── Conversation selector ────────────────────────────────────────────
    thread_id, _ = select_conversation(user_id=user_id)

    print(f"🌍 Travel Assistant with Memory (Snowflake Cortex + Redis)")
    print(f"   user={user_id}  thread={thread_id}")
    print("   Commands: exit | quit | debug | history | memories\n")

    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}, "recursion_limit": 50}
    state  = RuntimeState(messages=[], tool_call_count=0)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            save_transcript(state, thread_id, user_id)
            break

        if not user_input:
            continue

        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            save_transcript(state, thread_id, user_id)
            break

        if user_input.lower() == "debug":
            verbose = not verbose
            print(f"  [verbose mode {'ON' if verbose else 'OFF'}]\n")
            continue

        if user_input.lower() == "history":
            print_transcript(thread_id, user_id)
            continue

        if user_input.lower() == "memories":
            keys = redis_client.keys("memory:*")
            print(f"\n🧠 Long-term memories ({len(keys)} total):")
            for key in sorted(keys):
                data = redis_client.json().get(key)
                if data:
                    ts  = data.get("created_at", "")[:19]
                    mt  = data.get("memory_type", "?")
                    cnt = data.get("content", "")
                    print(f"  [{mt:8s}] {ts}  {cnt}")
            print()
            continue

        # ── Reset tool-call counter for new turn ─────────────────────────
        state["tool_call_count"] = 0
        state["messages"].append(HumanMessage(content=user_input))

        if verbose:
            ctx = get_memory_context(user_input, user_id=user_id, limit=5)
            print(f"── DEBUG memory context ──\n{ctx or '(none retrieved)'}\n──────────────────────────\n")

        try:
            for result in graph.stream(state, config=config, stream_mode="values"):
                state = RuntimeState(**result)

            # Find the last clean assistant reply
            reply = None
            for m in reversed(state["messages"]):
                if isinstance(m, AIMessage):
                    content = m.content.strip() if m.content else ""
                    if content and not extract_tool_call(content):
                        reply = content
                        break

            print(f"\nAssistant: {reply if reply else '(no reply — try again)'}\n")
            save_transcript(state, thread_id, user_id)

            if verbose:
                print(f"── DEBUG (tool_call_count={state.get('tool_call_count', 0)}) ──")
                for i, m in enumerate(state["messages"]):
                    kind    = type(m).__name__
                    content = (m.content or "")[:120].replace("\n", " ")
                    tc      = f" [calls={[t['name'] for t in getattr(m, 'tool_calls', [])]}]" if getattr(m, "tool_calls", None) else ""
                    print(f"  [{i:02d}] {kind}{tc}: {content!r}")
                print("──────────────────────────────────────\n")

        except Exception as e:
            logger.error(f"Error during graph execution: {e}", exc_info=True)
            print(f"\n⚠️  Error: {e}\n")

    return state


if __name__ == "__main__":
    uid = input("Enter user ID (default demo_user): ") or "demo_user"
    final_state = main(user_id=uid, verbose=False)


# Cell 14 – Inspect conversations (optional)

In [ ]:
# List all transcripts for demo_user
keys = redis_client.keys("conversation:demo_user:*")
print("Transcripts:")
for k in sorted(keys):
    print(k.decode())


In [ ]:
# View a specific transcript — replace the thread id below
# print_transcript("thread_20260807_054921", "demo_user")


In [ ]:
# List all long-term memories (without embedding vectors)
keys = redis_client.keys("memory:*")
print(f"Total memories: {len(keys)}")
for key in sorted(keys):
    data = redis_client.json().get(key)
    if data:
        print(json.dumps({k: v for k, v in data.items() if k != "embedding"}, indent=2))
        print("-" * 40)
